# Repository guide: Generic XLS-R SpecAugment blank-collapse diagnostic

Original code and recorded outputs retained. Read ../../docs/RUNNING.md before execution.


# XLS-R-300M → Tarifit V1.2 + SpecAugment

Full V1.2 experiment using `facebook/wav2vec2-xls-r-300m`.

- 1,754 frozen training segments / 129 frozen validation segments
- exact existing 34-token Tarifit tokenizer
- fresh Tarifit CTC head
- convolutional feature encoder frozen
- Transformer encoder fine-tuned
- SpecAugment: `mask_time_prob=0.05`, `mask_time_length=5`, no feature masking
- no speed perturbation
- LR `3e-5`, max 8 epochs, early stopping patience 2
- best checkpoint selected by validation CER


In [ ]:
# Cell 1 — Install dependencies
!pip -q install "transformers==4.57.1" "datasets==4.4.1" "accelerate>=1.10,<2" "jiwer==4.0.0" "safetensors>=0.4.5" "soundfile>=0.12.1"
print("✓ Dependencies installed.")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 116.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 511.6/511.6 kB 47.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 41.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 111.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.
✓ Dependencies installed.


In [ ]:
# Cell 2 — Mount Drive and define paths
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path

PROJECT_ROOT = Path("/content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm")
METADATA_PATH = PROJECT_ROOT / "data" / "metadata" / "segments_metadata_v1_2.csv"
FROZEN_METADATA_PATH = PROJECT_ROOT / "data" / "metadata" / "segments_metadata_v1_2_train_val_frozen.csv"
TOKENIZER_DIR = PROJECT_ROOT / "data" / "processed" / "mms_tokenizer_v1_2"
DATASET_CACHE_DIR = PROJECT_ROOT / "data" / "processed" / "mms_corpus_v1_2"
CACHE_MANIFEST_PATH = DATASET_CACHE_DIR / "cache_manifest.json"
OUTPUT_DIR = PROJECT_ROOT / "models" / "xlsr_300m_tarifit_v1_2_specaug"
RESULTS_DIR = PROJECT_ROOT / "results" / "xlsr_300m_tarifit_v1_2_specaug"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
BASE_MODEL_ID = "facebook/wav2vec2-xls-r-300m"
print("Base model:", BASE_MODEL_ID)
print("Output:", OUTPUT_DIR)


Mounted at /content/drive
Base model: facebook/wav2vec2-xls-r-300m
Output: /content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/models/xlsr_300m_tarifit_v1_2_specaug


In [ ]:
# Cell 3 — Verify versions, GPU, and seed
import json, random, hashlib
import numpy as np
import pandas as pd
import torch
import transformers, datasets

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("CUDA:", torch.cuda.is_available())
assert transformers.__version__ == "4.57.1"
assert datasets.__version__ == "4.4.1"
if not torch.cuda.is_available(): raise RuntimeError("Switch Colab to a GPU runtime.")
print("GPU:", torch.cuda.get_device_name(0))


Transformers: 4.57.1
Datasets: 4.4.1
CUDA: True
GPU: Tesla T4


In [ ]:
# Cell 4 — Load the exact frozen V1.2 split
frozen_df = pd.read_csv(FROZEN_METADATA_PATH)
for col in ["segment_id","recording_id","speaker_group_id","dataset_split","transcription"]:
    frozen_df[col] = frozen_df[col].fillna("").astype(str).str.strip()
frozen_df["dataset_split"] = frozen_df["dataset_split"].str.lower()
frozen_df["duration_seconds"] = pd.to_numeric(frozen_df["duration_seconds"], errors="raise")

train_df = frozen_df[frozen_df["dataset_split"].eq("train")].copy()
val_df = frozen_df[frozen_df["dataset_split"].eq("validation")].copy()
assert len(train_df) == 1754
assert len(val_df) == 129
assert train_df["segment_id"].is_unique and val_df["segment_id"].is_unique

print("Train:", len(train_df), "segments /", round(train_df["duration_seconds"].sum()/3600,3), "h")
print("Validation:", len(val_df), "segments /", round(val_df["duration_seconds"].sum()/3600,3), "h")
print("Train speakers:", sorted(train_df["speaker_group_id"].unique()))
print("Validation speakers:", sorted(val_df["speaker_group_id"].unique()))


Train: 1754 segments / 5.223 h
Validation: 129 segments / 0.298 h
Train speakers: ['SPK001', 'SPK002', 'SPK009']
Validation speakers: ['SPK007', 'SPK010']


In [ ]:
# Cell 5 — Verify speaker independence and held-out test isolation
train_speakers=set(train_df["speaker_group_id"].dropna())
val_speakers=set(val_df["speaker_group_id"].dropna())
assert not (train_speakers & val_speakers)

master_df=pd.read_csv(METADATA_PATH)
for col in ["dataset_split","speaker_group_id"]:
    master_df[col]=master_df[col].fillna("").astype(str).str.strip()
test_speakers=set(master_df.loc[master_df["dataset_split"].str.lower().eq("test"),"speaker_group_id"])
print("Train/validation overlap:", bool(train_speakers & val_speakers))
print("Train/test overlap:", bool(train_speakers & test_speakers))
print("Validation/test overlap:", bool(val_speakers & test_speakers))
assert not (train_speakers & test_speakers)
assert not (val_speakers & test_speakers)
print("✓ No speaker leakage.")


Train/validation overlap: False
Train/test overlap: False
Validation/test overlap: False
✓ No speaker leakage.


In [ ]:
# Cell 6 — Load the exact existing V1.2 tokenizer
from transformers import Wav2Vec2CTCTokenizer, Wav2Vec2FeatureExtractor, Wav2Vec2Processor

FINAL_LETTERS=["a","b","c","d","ḍ","e","ɛ","f","g","h","ḥ","i","j","k","l","m","n","p","q","r","s","t","ṭ","u","v","w","x","y","z","ɣ","ʷ"]
VOCAB_PATH = TOKENIZER_DIR / "vocab.json"
with open(VOCAB_PATH,"r",encoding="utf-8") as f: existing_vocab=json.load(f)
expected_tokens=set(FINAL_LETTERS)|{"|","[UNK]","[PAD]"}
assert len(existing_vocab)==34
assert set(existing_vocab.keys())==expected_tokens
assert sorted(existing_vocab.values())==list(range(34))

tokenizer=Wav2Vec2CTCTokenizer(vocab_file=str(VOCAB_PATH),unk_token="[UNK]",pad_token="[PAD]",word_delimiter_token="|",bos_token=None,eos_token=None,do_lower_case=False)
feature_extractor=Wav2Vec2FeatureExtractor(feature_size=1,sampling_rate=16000,padding_value=0.0,do_normalize=True,return_attention_mask=True)
processor=Wav2Vec2Processor(feature_extractor=feature_extractor,tokenizer=tokenizer)
print("Tokenizer size:", len(tokenizer))
print("PAD id:", tokenizer.pad_token_id)
print("UNK id:", tokenizer.unk_token_id)


Tokenizer size: 34
PAD id: 33
UNK id: 32


In [ ]:
# Cell 7 — Load cached V1.2 waveforms and labels
from datasets import load_from_disk

metadata_sha256=hashlib.sha256(FROZEN_METADATA_PATH.read_bytes()).hexdigest()
with open(CACHE_MANIFEST_PATH,"r",encoding="utf-8") as f: cache_manifest=json.load(f)
assert cache_manifest.get("metadata_sha256")==metadata_sha256

dataset=load_from_disk(str(DATASET_CACHE_DIR))
assert len(dataset["train"])==1754
assert len(dataset["validation"])==129
print("Cached train:", len(dataset["train"]))
print("Cached validation:", len(dataset["validation"]))
print("Columns:", dataset["train"].column_names)
print("Cached train hours:", round(sum(dataset["train"]["input_length"])/16000/3600,3))


Cached train: 1754
Cached validation: 129
Columns: ['segment_id', 'input_values', 'input_length', 'labels']
Cached train hours: 5.223


In [ ]:
# Cell 8 — Verify tokenizer coverage
unk_id=tokenizer.unk_token_id
unknown_train=[i for i,labels in enumerate(dataset["train"]["labels"]) if unk_id in labels]
unknown_val=[i for i,labels in enumerate(dataset["validation"]["labels"]) if unk_id in labels]
print("Train with [UNK]:", len(unknown_train))
print("Validation with [UNK]:", len(unknown_val))
assert not unknown_train
assert not unknown_val


Train with [UNK]: 0
Validation with [UNK]: 0


In [ ]:
# Cell 9 — Load XLS-R-300M with fresh Tarifit CTC head and SpecAugment
from transformers import Wav2Vec2ForCTC

MASK_TIME_PROB=0.05
MASK_TIME_LENGTH=5
MASK_FEATURE_PROB=0.0

model=Wav2Vec2ForCTC.from_pretrained(
    BASE_MODEL_ID,
    vocab_size=len(tokenizer),
    pad_token_id=tokenizer.pad_token_id,
    ctc_loss_reduction="mean",
    ctc_zero_infinity=True,
    ignore_mismatched_sizes=True,
    apply_spec_augment=True,
    mask_time_prob=MASK_TIME_PROB,
    mask_time_length=MASK_TIME_LENGTH,
    mask_feature_prob=MASK_FEATURE_PROB,
    attention_dropout=0.05,
    hidden_dropout=0.05,
    feat_proj_dropout=0.0,
    layerdrop=0.0,
)
model.freeze_feature_encoder()
model.config.use_cache=False

total_params=sum(p.numel() for p in model.parameters())
trainable_params=sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Trainable percentage: {100*trainable_params/total_params:.4f}%")
print("CTC vocab:", model.config.vocab_size)
print("SpecAugment:", model.config.apply_spec_augment)
print("mask_time_prob:", model.config.mask_time_prob)
print("mask_time_length:", model.config.mask_time_length)
print("mask_feature_prob:", model.config.mask_feature_prob)
assert model.config.vocab_size==34
assert model.config.apply_spec_augment is True
assert abs(model.config.mask_time_prob-0.05)<1e-12
assert model.config.mask_time_length==5
assert model.config.mask_feature_prob==0.0


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.27G [00:00<?, ?B/s]

Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at facebook/wav2vec2-xls-r-300m and are newly initialized: ['lm_head.bias', 'lm_head.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Total parameters: 315,473,570
Trainable parameters: 311,263,394
Trainable percentage: 98.6654%
CTC vocab: 34
SpecAugment: True
mask_time_prob: 0.05
mask_time_length: 5
mask_feature_prob: 0.0


In [ ]:
# Cell 10 — Check CTC feasibility
def minimum_ctc_frames(labels):
    labels=list(labels)
    repeats=sum(labels[i]==labels[i-1] for i in range(1,len(labels)))
    return len(labels)+repeats

def output_frames(input_samples):
    return int(model._get_feat_extract_output_lengths(torch.tensor(int(input_samples))).item())

def find_bad(split_ds):
    bad=[]
    for i,ex in enumerate(split_ds):
        out=output_frames(ex["input_length"])
        need=minimum_ctc_frames(ex["labels"])
        if out<need:
            bad.append({"index":i,"segment_id":ex["segment_id"],"output_frames":out,"minimum_ctc_frames":need})
    return bad

bad_train=find_bad(dataset["train"])
bad_val=find_bad(dataset["validation"])
print("CTC-infeasible train:", len(bad_train))
print("CTC-infeasible validation:", len(bad_val))
if bad_train: display(pd.DataFrame(bad_train))
if bad_val: display(pd.DataFrame(bad_val))
assert not bad_train
assert not bad_val


model.safetensors:   0%|          | 0.00/1.27G [00:00<?, ?B/s]

CTC-infeasible train: 0
CTC-infeasible validation: 0


In [ ]:
# Cell 11 — Define dynamic CTC padding
from dataclasses import dataclass
from typing import Any, Union

@dataclass
class DataCollatorCTCWithPadding:
    processor: Any
    padding: Union[bool,str]=True

    def __call__(self,features):
        input_features=[{"input_values":f["input_values"]} for f in features]
        label_features=[{"input_ids":f["labels"]} for f in features]
        batch=self.processor.pad(input_features,padding=self.padding,return_tensors="pt")
        labels_batch=self.processor.tokenizer.pad(label_features,padding=self.padding,return_tensors="pt")
        batch["labels"]=labels_batch["input_ids"].masked_fill(labels_batch["attention_mask"].ne(1),-100)
        return batch

data_collator=DataCollatorCTCWithPadding(processor=processor)
print("✓ Collator ready.")


✓ Collator ready.


In [ ]:
# Cell 12 — Define WER and CER
from jiwer import wer, cer

def compute_metrics(pred):
    pred_ids=np.argmax(pred.predictions,axis=-1)
    label_ids=pred.label_ids.copy()
    label_ids[label_ids==-100]=tokenizer.pad_token_id
    pred_str=[x.strip() for x in processor.batch_decode(pred_ids)]
    ref_str=[x.strip() for x in processor.batch_decode(label_ids,group_tokens=False)]
    return {"wer":wer(ref_str,pred_str),"cer":cer(ref_str,pred_str)}

print("✓ Metrics ready.")


✓ Metrics ready.


In [ ]:
# Cell 13 — Configure XLS-R training
from transformers import TrainingArguments, EarlyStoppingCallback

training_args=TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=8,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=3e-5,
    weight_decay=0.01,
    warmup_steps=100,
    lr_scheduler_type="linear",
    fp16=torch.cuda.is_available(),
    gradient_checkpointing=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=25,
    load_best_model_at_end=True,
    metric_for_best_model="cer",
    greater_is_better=False,
    save_total_limit=2,
    report_to="none",
    seed=SEED,
    data_seed=SEED,
    dataloader_num_workers=2,
    remove_unused_columns=False,
)

early_stopping=EarlyStoppingCallback(early_stopping_patience=2,early_stopping_threshold=0.001)
print("Train examples:", len(dataset["train"]))
print("Validation examples:", len(dataset["validation"]))
print("Learning rate:", training_args.learning_rate)
print("Effective batch size:", 2*8)
print("Max epochs:", training_args.num_train_epochs)
print("SpecAugment:", model.config.apply_spec_augment)


Train examples: 1754
Validation examples: 129
Learning rate: 3e-05
Effective batch size: 16
Max epochs: 8
SpecAugment: True


In [ ]:
# Cell 14 — Create Trainer
from transformers import Trainer

trainer=Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=processor.feature_extractor,
    callbacks=[early_stopping],
)
print("✓ Trainer ready.")


✓ Trainer ready.


In [ ]:
# Cell 15 — Start or resume training
from transformers.trainer_utils import get_last_checkpoint

last_checkpoint=get_last_checkpoint(str(OUTPUT_DIR)) if OUTPUT_DIR.exists() else None
if last_checkpoint:
    print("Resuming from:", last_checkpoint)
    train_result=trainer.train(resume_from_checkpoint=last_checkpoint)
else:
    print("Starting fresh XLS-R + SpecAugment training.")
    train_result=trainer.train()

print("Training finished.")
print("Best checkpoint:", trainer.state.best_model_checkpoint)
print("Best validation CER:", trainer.state.best_metric)


Starting fresh XLS-R + SpecAugment training.


/usr/local/lib/python3.13/dist-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)


Epoch,Training Loss,Validation Loss,Wer,Cer
1,4.338400,3.997478,1.000000,1.000000
2,3.103400,3.321098,1.000000,1.000000
3,2.953200,3.265681,1.000000,1.000000


/usr/local/lib/python3.13/dist-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)
/usr/local/lib/python3.13/dist-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)


Training finished.
Best checkpoint: /content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/models/xlsr_300m_tarifit_v1_2_specaug/checkpoint-110
Best validation CER: 1.0


In [ ]:
# Cell 16 — Show and save epoch-by-epoch results
rows=[]
last_train_loss=None
for log in trainer.state.log_history:
    if "loss" in log and "eval_loss" not in log:
        last_train_loss=log["loss"]
    if "eval_loss" in log:
        rows.append({
            "epoch":log.get("epoch"),
            "training_loss":last_train_loss,
            "validation_loss":log.get("eval_loss"),
            "WER":log.get("eval_wer"),
            "CER":log.get("eval_cer"),
        })

history_df=pd.DataFrame(rows)
display(history_df)
history_df.to_csv(RESULTS_DIR/"training_history.csv",index=False)


,epoch,training_loss,validation_loss,WER,CER
0,1.0,4.3384,3.997478,1.0,1.0
1,2.0,3.1034,3.321098,1.0,1.0
2,3.0,2.9532,3.265681,1.0,1.0


In [ ]:
# Cell 17 — Evaluate and save CER-selected best checkpoint
best_metrics=trainer.evaluate(eval_dataset=dataset["validation"])
best_wer=float(best_metrics["eval_wer"])
best_cer=float(best_metrics["eval_cer"])
print(f"Best validation WER: {best_wer:.6f} ({best_wer*100:.2f}%)")
print(f"Best validation CER: {best_cer:.6f} ({best_cer*100:.2f}%)")
print("Selected checkpoint:", trainer.state.best_model_checkpoint)

BEST_MODEL_DIR=OUTPUT_DIR/"best_model"
trainer.save_model(str(BEST_MODEL_DIR))
processor.save_pretrained(str(BEST_MODEL_DIR))
print("Saved:", BEST_MODEL_DIR)


Best validation WER: 1.000000 (100.00%)
Best validation CER: 1.000000 (100.00%)
Selected checkpoint: /content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/models/xlsr_300m_tarifit_v1_2_specaug/checkpoint-110
Saved: /content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/models/xlsr_300m_tarifit_v1_2_specaug/best_model


In [ ]:
# Cell 18 — Save predictions and compute space-insensitive CER
prediction_output=trainer.predict(dataset["validation"])
pred_ids=np.argmax(prediction_output.predictions,axis=-1)
label_ids=prediction_output.label_ids.copy()
label_ids[label_ids==-100]=tokenizer.pad_token_id

predictions=[x.strip() for x in processor.batch_decode(pred_ids)]
references=[x.strip() for x in processor.batch_decode(label_ids,group_tokens=False)]

prediction_df=pd.DataFrame({
    "segment_id":dataset["validation"]["segment_id"],
    "reference":references,
    "prediction":predictions,
})
prediction_df["segment_wer"]=[wer(r,p) for r,p in zip(references,predictions)]
prediction_df["segment_cer"]=[cer(r,p) for r,p in zip(references,predictions)]
prediction_df["segment_cer_no_spaces"]=[cer(r.replace(" ",""),p.replace(" ","")) for r,p in zip(references,predictions)]
global_cer_no_spaces=cer([r.replace(" ","") for r in references],[p.replace(" ","") for p in predictions])
prediction_df.to_csv(RESULTS_DIR/"validation_predictions.csv",index=False,encoding="utf-8")
display(prediction_df.head(20))
print("Standard CER:", f"{best_cer*100:.2f}%")
print("Space-insensitive CER:", f"{global_cer_no_spaces*100:.2f}%")


,segment_id,reference,prediction,segment_wer,segment_cer,segment_cer_no_spaces
0,REC090_SEG0010,ssalamuɛlikum necc meryem,,1.0,1.0,1.0
1,REC090_SEG0011,aqay ruxxa tnayn uɛecrin sana ḍi hulanda,,1.0,1.0,1.0
2,REC090_SEG0012,mercex ak nmis n jjiran usiɣ ḍ zi lmeɣrib umi ...,,1.0,1.0,1.0
3,REC090_SEG0013,umi wsiɣd dda ufix manayenni wa dji ca min ira...,,1.0,1.0,1.0
4,REC090_SEG0014,a necc mammec ira djjix ḍi lmeɣrib wadji manay...,,1.0,1.0,1.0
5,REC090_SEG0015,necc ḍi lmeɣrib ira ɣari lḥurriya inu ira ɣari...,,1.0,1.0,1.0
6,REC090_SEG0016,ḍi lmeɣrib neccin mammec ira niɛicc,,1.0,1.0,1.0
7,REC090_SEG0017,ak baba dd yemma wa ɣaneɣ ca ɣaneɣ ca n reḥway...,,1.0,1.0,1.0
8,REC090_SEG0018,lmuhim wsiɣd,,1.0,1.0,1.0
9,REC090_SEG0019,necc ira ɛemmas wa wsiɣd ɣar urupa wsiɣd ḍi ṭi...,,1.0,1.0,1.0


Standard CER: 100.00%
Space-insensitive CER: 100.00%


In [ ]:
# Cell 19 — Compare with current V1.2 systems
comparison_df=pd.DataFrame([
    {"model":"MMS-1B + SpecAugment","configuration":"Full V1.2","WER":0.849915,"CER":0.432671},
    {"model":"MMS-1B + SpecAugment","configuration":"Half-Bible V1.2","WER":0.847364,"CER":0.424954},
    {"model":"OmniASR-W2V-300M","configuration":"Full V1.2, no augmentation","WER":0.910714,"CER":0.457834},
    {"model":"Fadhma-300M","configuration":"Full V1.2, no augmentation","WER":0.896259,"CER":0.462497},
    {"model":"XLS-R-300M + SpecAugment","configuration":"Full V1.2","WER":best_wer,"CER":best_cer},
])
comparison_df["WER_percent"]=100*comparison_df["WER"]
comparison_df["CER_percent"]=100*comparison_df["CER"]
display(comparison_df)
comparison_df.to_csv(RESULTS_DIR/"current_model_comparison.csv",index=False)
print("Note: augmentation differs across some rows; state this explicitly in the thesis.")


,model,configuration,WER,CER,WER_percent,CER_percent
0,MMS-1B + SpecAugment,Full V1.2,0.849915,0.432671,84.9915,43.2671
1,MMS-1B + SpecAugment,Half-Bible V1.2,0.847364,0.424954,84.7364,42.4954
2,OmniASR-W2V-300M,"Full V1.2, no augmentation",0.910714,0.457834,91.0714,45.7834
3,Fadhma-300M,"Full V1.2, no augmentation",0.896259,0.462497,89.6259,46.2497
4,XLS-R-300M + SpecAugment,Full V1.2,1.000000,1.000000,100.0000,100.0000


Note: augmentation differs across some rows; state this explicitly in the thesis.


In [ ]:
# Cell 20 — Save experiment summary
summary={
    "experiment":"XLS-R-300M to Tarifit V1.2 with SpecAugment",
    "base_model":BASE_MODEL_ID,
    "seed":SEED,
    "train_segments":int(len(train_df)),
    "train_hours":float(train_df["duration_seconds"].sum()/3600),
    "validation_segments":int(len(val_df)),
    "validation_hours":float(val_df["duration_seconds"].sum()/3600),
    "tokenizer_size":int(len(tokenizer)),
    "feature_extractor_frozen":True,
    "encoder_finetuned":True,
    "fresh_tarifit_ctc_head":True,
    "augmentation":{
        "specaugment":True,
        "mask_time_prob":MASK_TIME_PROB,
        "mask_time_length":MASK_TIME_LENGTH,
        "mask_feature_prob":MASK_FEATURE_PROB,
        "speed_perturbation":False,
    },
    "total_parameters":int(total_params),
    "trainable_parameters":int(trainable_params),
    "learning_rate":3e-5,
    "weight_decay":0.01,
    "warmup_steps":100,
    "max_epochs":8,
    "physical_batch_size":2,
    "gradient_accumulation_steps":8,
    "effective_batch_size":16,
    "early_stopping_patience":2,
    "checkpoint_selection_metric":"CER",
    "best_checkpoint":trainer.state.best_model_checkpoint,
    "best_validation_wer":best_wer,
    "best_validation_cer":best_cer,
    "space_insensitive_validation_cer":float(global_cer_no_spaces),
}
with open(RESULTS_DIR/"experiment_summary.json","w",encoding="utf-8") as f:
    json.dump(summary,f,ensure_ascii=False,indent=2)
print(json.dumps(summary,indent=2))


{
  "experiment": "XLS-R-300M to Tarifit V1.2 with SpecAugment",
  "base_model": "facebook/wav2vec2-xls-r-300m",
  "seed": 42,
  "train_segments": 1754,
  "train_hours": 5.222733333333333,
  "validation_segments": 129,
  "validation_hours": 0.2983577777777778,
  "tokenizer_size": 34,
  "feature_extractor_frozen": true,
  "encoder_finetuned": true,
  "fresh_tarifit_ctc_head": true,
  "augmentation": {
    "specaugment": true,
    "mask_time_prob": 0.05,
    "mask_time_length": 5,
    "mask_feature_prob": 0.0,
    "speed_perturbation": false
  },
  "total_parameters": 315473570,
  "trainable_parameters": 311263394,
  "learning_rate": 3e-05,
  "weight_decay": 0.01,
  "warmup_steps": 100,
  "max_epochs": 8,
  "physical_batch_size": 2,
  "gradient_accumulation_steps": 8,
  "effective_batch_size": 16,
  "early_stopping_patience": 2,
  "checkpoint_selection_metric": "CER",
  "best_checkpoint": "/content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/models/xlsr_300m_tarifit_v1_2_specaug/

## Interpretation

Use the CER-selected checkpoint as the official XLS-R validation result.

This is a practical low-resource configuration with SpecAugment. When comparing it with OmniASR or Fadhma, state explicitly that those full-data runs did not use augmentation, so that comparison reflects the complete system configuration rather than a perfectly isolated architecture-only ablation.
